# Mission Eagle - 1: Lunar Lander Automatic Landing 🚀

The goal of this mission is to develop an autopilot capable of safely landing the lunar module on its target zone.

## Phase 1: Exploration and Baseline Model
Before optimizing, we will:
1. Explore the `LunarLander-v3` environment.
2. Evaluate the performance of an untrained/default model.
3. Train a "Baseline" agent using the default PPO algorithm.
4. Evaluate this baseline model to establish a starting point.


In [ ]:
import sys
import os
import logging
import gymnasium as gym

# Configure logging to output cleanly to the notebook
logging.basicConfig(level=logging.INFO, stream=sys.stdout, format='%(message)s')
logger = logging.getLogger(__name__)

# Add the parent directory to the path so we can import from src
sys.path.append(os.path.abspath('..'))

from src.mission.training import train_baseline, evaluate_agent
from src.mission.visualization import show_agent_flight
from IPython.display import Video

### 1. Environment Exploration
Let's analyze the possible actions and observations the agent can interact with.

In [ ]:
env = gym.make("LunarLander-v3")

logger.info(f"Action space: {env.action_space}")
logger.info(f"Observation space: {env.observation_space}")

# The actions are discrete (0 to 3):
# 0: Do nothing
# 1: Fire left orientation engine
# 2: Fire main engine
# 3: Fire right orientation engine

Action space: Discrete(4)
Observation space: Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)


### 2. Training a Baseline Agent
We will use the PPO (Proximal Policy Optimization) algorithm. The logic is hidden in `src/mission/training.py` to keep the notebook clean.

In [ ]:
# Train the baseline model for 50,000 steps
model, env = train_baseline("LunarLander-v3", timesteps=50000)

Starting baseline training for 50000 timesteps...
Baseline training complete.


### 3. Evaluating the Baseline
Now that the baseline model is trained, let's measure its average performance over 50 episodes.

In [ ]:
# Evaluate the agent
evaluate_agent(model, env, n_episodes=50)

# A successful landing is considered > 200 points. We'll see where we stand.

Evaluating the model over 50 episodes...


/Users/asmaaliouche/OC/7. Reinforcement Learning/.venv/lib/python3.10/site-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Performance: Mean reward = -147.73 +/- 25.63


(np.float64(-147.72768177220598), np.float64(25.63071916005262))

### 4. Visualizing the Baseline Agent
To truly understand how bad a score of -200 is, let's watch the agent attempt a landing. We use a function from `src/mission/visualization.py` to render the video.

In [ ]:
# Record and display a single flight
show_agent_flight("LunarLander-v3", model)

Recording the agent's flight...
Rendering video...
Animation.save using <class 'matplotlib.animation.HTMLWriter'>


## Phase 2: Agent Optimization
The baseline agent exhibited poor performance with an average reward of roughly -370. Our objective for this phase is to achieve and sustain an average reward exceeding the 200-point threshold, indicating a consistently successful landing.

To achieve this, we conducted a systematic hyperparameter tuning process. Rather than modifying the environment, we adjusted the parameters governing the PPO algorithm's learning dynamics. Experiments were tracked and logged using TensorBoard to facilitate the comparative analysis of learning curves. (To view the training metrics, run `tensorboard --logdir logs/mission_eagle/` in your terminal).

Below is the demonstration of our final optimized model's performance.

In [ ]:
Video("eagle_landing.mp4", embed=True)

### Hyperparameter Tuning Summary and Interpretation

We tested isolated hyperparameter adjustments before leveraging a comprehensive configuration sourced from the RL Baselines Zoo.

**1. Experiment: PPO_Opt_Batch128**
- **Modification:** Increased `batch_size` to 128 (default: 64).
- **Result:** Mean reward of 1.16.
- **Interpretation:** Increasing the batch size allows the agent to aggregate more experiences before performing a policy update, slightly stabilizing the learning trajectory. However, this adjustment alone was insufficient to solve the environment.

**2. Experiment: PPO_Opt_LR_0001**
- **Modification:** Decreased `learning_rate` to 0.0001 (default: 0.0003).
- **Result:** Mean reward of -51.70.
- **Interpretation:** The reduced learning rate caused the policy updates to be too conservative. Consequently, the agent failed to converge on a viable landing strategy within the allocated timesteps.

**3. Experiment: PPO_Opt_RLZoo**
- **Modification:** Applied the tuned PPO hyperparameters from the RL Baselines Zoo.
- **Result:** Mean reward of 132.80.
- **Interpretation:** Leveraging community-optimized hyperparameters significantly accelerated the learning process. The agent successfully landed in many episodes but lacked the consistency required to breach the 200-point threshold.

**4. Final Experiment: PPO_Opt_RLZoo_800k**
- **Modification:** Retained the RL Zoo hyperparameters but extended the training duration to 800,000 timesteps.
- **Result:** Mean reward of **225.78**.
- **Interpretation:** Providing the agent with an extended training horizon under an optimal hyperparameter configuration allowed the policy to fully converge. The agent now demonstrates a highly stable and successful landing behavior, reliably exceeding the 200-point target.

**Conclusion:** The optimal model has been serialized to `final_eagle_model.zip` and is ready for deployment in Phase 3.

## Phase 3: Deployment and Interface
In this final phase, we have deployed the optimized RL model using a **FastAPI** backend and created an interactive dashboard using **Streamlit**.

### Architecture
- **API (Backend):** Built with FastAPI (`src/mission/api/app.py`). It loads the trained `final_eagle_model.zip`, exposes a `/play` endpoint, receives the environment state, and returns the predicted action.
- **GUI & Dashboard (Frontend):** Built with Streamlit (`src/mission/gui/dashboard.py`). It renders the LunarLander environment, communicates with the API to get actions, displays the live animation, and tracks real-time telemetry (rewards, steps, actions).

### Instructions to Run
Open two separate terminal windows in the root of the project:

**Terminal 1 (Start the API):**
```bash
source .venv/bin/activate
uvicorn src.mission.api.app:app --reload
```

**Terminal 2 (Start the Dashboard):**
```bash
source .venv/bin/activate
streamlit run src/mission/gui/dashboard.py
```